In [ ]:
!git clone https://github.com/jovajara/helium-orbitals.git

In [ ]:
%cd helium-orbitals/
!make clean
!make
!./hf_helium

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import numpy as np
import glob
import struct

# Buscar archivos binarios
files = sorted(glob.glob("helium_*.bin"))

if not files:
    print("No se encontraron archivos .bin. ¡Ejecuta primero la simulación en C!")
else:
    print(f"Procesando {len(files)} archivos (1s -> 3d)...\n")

    for filename in files:
        name_clean = filename.replace("helium_", "").replace(".bin", "").upper()
        print(f"   -> Graficando: {name_clean}...")

        try:
            # 1. LEER BINARIO
            with open(filename, "rb") as f:
                N = struct.unpack('i', f.read(4))[0]
                box_size = struct.unpack('d', f.read(8))[0]
                data = np.fromfile(f, dtype=np.float64, count=-1)

            # Reformatear a 3D (Orden C)
            vol_data = data.reshape((N, N, N), order='C')

            # 2. SELECCIONAR CORTE (Siempre Z=0)
            mid = N // 2
            slice_2d = vol_data[:, :, mid].T # Transponer para orientación correcta

            # 3. GRAFICAR
            fig, ax = plt.subplots(figsize=(8, 7))

            # Título
            ax.set_title(f"Orbital Helio {name_clean} (Corte Z=0)\nCaja: {box_size:.1f} a.u.",
                         fontsize=15, fontweight='bold', pad=12)

            extent = [-box_size, box_size, -box_size, box_size]
            dens_max = slice_2d.max()

            # Mapa de color
            im = ax.imshow(slice_2d,
                           cmap='inferno',
                           origin='lower',
                           extent=extent,
                           norm=colors.LogNorm(vmin=max(1e-12, dens_max*1e-5), vmax=dens_max))

            ax.set_xlabel("X (u.a.)", fontsize=12)
            ax.set_ylabel("Y (u.a.)", fontsize=12)
            cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            cbar.set_label("Densidad (Log)", fontsize=12)

            plt.tight_layout()
            savename = f"vis_{name_clean.lower()}.png"
            plt.savefig(savename, dpi=150)
            print(f"      Guardado: {savename}")
            plt.show()
            plt.close()

        except Exception as e:
            print(f"Error procesando {filename}: {e}")